# Day 6 practice — One waiter, many tables

**Read first:** [11_theory_async_basics.md](11_theory_async_basics.md)

You'll measure sequential versus concurrent calls with a stopwatch, then **reproduce the blocked
event loop** and watch a second request wait behind the first. That second experiment is the reason
this day exists.

No database and no network needed — `asyncio.sleep` stands in for I/O perfectly.

In [ ]:
import asyncio
import time
from concurrent.futures import ThreadPoolExecutor

from fastapi import FastAPI
from fastapi.testclient import TestClient

class Stopwatch:
    def __enter__(self):
        self.t0 = time.perf_counter(); return self
    def __exit__(self, *exc):
        self.elapsed = time.perf_counter() - self.t0

async def timed(label, coro):
    # Await a coroutine and report how long it took.
    with Stopwatch() as sw:
        result = await coro
    print(f"{label:<44} {sw.elapsed:6.2f}s")
    return result

# NOTE: notebooks already run inside an asyncio event loop, so `asyncio.run(...)`
# raises "cannot be called from a running event loop". Use top-level `await`
# instead - Jupyter supports it directly in a cell. In a .py script you WOULD
# use asyncio.run() as the single entry point.
print("ready")

---

## Part 1 — A coroutine does nothing until you await it

In [ ]:
async def get_temp(city: str) -> float:
    await asyncio.sleep(0.3)          # stands in for "waiting on a database or an API"
    return 21.4

# Calling it does NOT run it.
maybe = get_temp("Utrecht")
print("calling it gives you:", maybe)
print("type                :", type(maybe).__name__)

# Awaiting it runs it.
print("awaiting it gives   :", await get_temp("Utrecht"))

maybe.close()      # tidy up the coroutine we created and never awaited

Python also warns you about this: `RuntimeWarning: coroutine 'get_temp' was never awaited`. When
an async function "does nothing", that warning is usually the entire bug.

> 🎯 **`async def` defines it. `await` runs it.**

---

## Part 2 — Sequential vs `gather`, measured

Six cities, each taking 0.3 s of *waiting*.

In [ ]:
CITIES = ["De Bilt", "Amsterdam", "Rotterdam", "Eindhoven", "Groningen", "Maastricht"]

async def sequential():
    return [await get_temp(c) for c in CITIES]      # await inside the loop = one at a time

async def concurrent():
    return await asyncio.gather(*(get_temp(c) for c in CITIES))   # start all, wait for all

await timed("sequential (await inside the loop)", sequential())
await timed("gather     (all six at once)",       concurrent())

Roughly **1.8 s versus 0.3 s**. Same six calls, same waiting, same machine.

Nothing got faster. You simply stopped standing at the pass watching each dish cook before taking
the next order.

### 🔮 Predict

`get_temp` does `await asyncio.sleep(0.3)` — it uses no CPU at all. What would happen if it did 0.3 s
of **real computation** instead, and you gathered six of those?

In [ ]:
def burn_cpu(seconds: float) -> int:
    end = time.perf_counter() + seconds
    n = 0
    while time.perf_counter() < end:
        n += 1
    return n

async def cpu_task():
    return burn_cpu(0.3)              # NOT awaiting anything - pure computation

async def gather_cpu():
    return await asyncio.gather(*(cpu_task() for _ in range(6)))

await timed("gather over CPU-BOUND work", gather_cpu())

**Still about 1.8 s.** `gather` gained nothing.

There was never a moment where a task was *waiting*, so there was never an opportunity to switch to
another one. Async buys you concurrency while waiting; it cannot manufacture a second CPU.

| Work | Async helps? | What does help |
|---|---|---|
| **I/O-bound** (database, HTTP, disk) | ✅ enormously | `async` + `await` |
| **CPU-bound** (parsing, resizing, training) | ❌ not at all | multiple **processes** |

---

## Part 3 — 🚨 Blocking the event loop

The most important experiment in this notebook. Four endpoints, each taking one second.

In [ ]:
app = FastAPI()

@app.get("/sync-blocking")           # def + blocking  -> threadpool. SAFE.
def sync_blocking():
    time.sleep(1)
    return {"handler": "sync-blocking"}

@app.get("/async-proper")            # async def + await -> event loop, yields. SAFE.
async def async_proper():
    await asyncio.sleep(1)
    return {"handler": "async-proper"}

@app.get("/async-blocking")          # async def + BLOCKING -> 🚨 freezes the loop
async def async_blocking():
    time.sleep(1)                    # no await. Nothing yields control.
    return {"handler": "async-blocking"}

@app.get("/quick")                   # the canary: should always be instant
def quick():
    return {"handler": "quick"}

client = TestClient(app)
print("all four endpoints registered")

### This needs a REAL server

`TestClient` gives each request its own event loop, so it physically cannot show loop starvation —
the very thing we want to observe. So we start a real uvicorn server in a background thread. One
process, **one event loop**, exactly like production.

In [ ]:
import threading

import requests
import uvicorn

PORT = 8099
config = uvicorn.Config(app, host="127.0.0.1", port=PORT, log_level="error")
server = uvicorn.Server(config)
thread = threading.Thread(target=server.run, daemon=True)
thread.start()

for _ in range(100):                       # wait for startup
    if server.started:
        break
    time.sleep(0.05)

BASE = f"http://127.0.0.1:{PORT}"
print("real uvicorn server running on", BASE)
print("sanity check:", requests.get(f"{BASE}/quick", timeout=5).json())

Now the experiment. We fire **one slow request** and, at the same moment, five quick ones. If the
slow endpoint is well behaved the quick ones return immediately. If it blocks the event loop, they
queue behind it.

### 🔮 Predict

Three runs below. For which one will `/quick` be slow — and roughly how slow?

In [ ]:
def hammer(slow_path: str, n_quick: int = 5):
    # Fire one slow request plus n quick ones at once; time how long the quick ones took.
    with ThreadPoolExecutor(max_workers=n_quick + 1) as pool:
        t0 = time.perf_counter()
        slow = pool.submit(requests.get, f"{BASE}{slow_path}", timeout=30)
        time.sleep(0.1)                                    # let the slow one get going
        quicks = [pool.submit(requests.get, f"{BASE}/quick", timeout=30) for _ in range(n_quick)]
        quick_times = [(f.result(), time.perf_counter() - t0) for f in quicks]
        slow.result()
        total = time.perf_counter() - t0

    worst_quick = max(t for _, t in quick_times)
    print(f"  slow endpoint : {slow_path}")
    print(f"  total wall    : {total:5.2f}s")
    print(f"  slowest /quick: {worst_quick:5.2f}s   <-- THE NUMBER THAT MATTERS")
    return worst_quick

print("A) def + time.sleep  (threadpool)")
a = hammer("/sync-blocking")
print()
print("B) async def + await asyncio.sleep  (proper async)")
b = hammer("/async-proper")
print()
print("C) async def + time.sleep  (🚨 blocking the loop)")
c = hammer("/async-blocking")

### Read those three numbers

In A and B the `/quick` requests came back in milliseconds. In **C** they took roughly a **full
second** — they were stuck behind a handler that never yielded control.

And `/quick` is a completely unrelated endpoint. So is your health check. So is every other request
in the entire service.

Note that A and C contain the **identical line** — `time.sleep(1)`. The keyword on the line above is
the whole difference:

| Handler | Runs on | Blocking code is |
|---|---|---|
| `def` | a **threadpool** thread | ✅ fine — it blocks one worker |
| `async def` | the **event loop** | 🚨 catastrophic — it blocks everything |

> 🎯 **If you're not `await`ing anything, use plain `def`.**

The cruel part: with one user, C looks perfectly fine. It fails only under load, in production.

In [ ]:
print("latency of an UNRELATED /quick request while the slow endpoint runs:")
print(f"  A  def + time.sleep        {a:5.2f}s   safe (threadpool)")
print(f"  B  async def + await       {b:5.2f}s   safe (loop stays free)")
print(f"  C  async def + time.sleep  {c:5.2f}s   <- blocked the loop")
print()
print(f"C made an unrelated endpoint {c / max(a, 0.01):.0f}x slower.")

# shut the server down cleanly
server.should_exit = True
thread.join(timeout=10)
print()
print("server stopped")

---

## Part 4 — Which library is which

The rule is only useful if you can classify your libraries.

| Library | Kind | Handler |
|---|---|---|
| `psycopg2`, SQLAlchemy (default engine) | blocking | `def` |
| `requests` | blocking | `def` |
| `time.sleep` | blocking | `def` |
| `httpx2.AsyncClient` | async | `async def` + `await` |
| `asyncpg`, SQLAlchemy async engine | async | `async def` + `await` |
| dict/list work, computation | neither | `def` |

**Your `insight-api` project uses `def` handlers throughout**, because it talks to Postgres through
psycopg2, which is blocking. That is the correct choice, not a shortcut.

### Mixing is allowed

In [ ]:
mixed = FastAPI()

def sync_dependency():               # blocking dependency -> threadpool
    time.sleep(0.05)
    return "from-sync-dep"

async def async_dependency():        # async dependency -> event loop
    await asyncio.sleep(0.05)
    return "from-async-dep"

from fastapi import Depends

@mixed.get("/a")
async def async_handler_sync_dep(v = Depends(sync_dependency)):
    return {"handler": "async", "dep": v}

@mixed.get("/b")
def sync_handler_async_dep(v = Depends(async_dependency)):
    return {"handler": "sync", "dep": v}

mc = TestClient(mixed)
print("async handler + sync dependency :", mc.get("/a").json())
print("sync handler + async dependency :", mc.get("/b").json())
print()
print("Both work. FastAPI looks at each function's OWN definition and routes it accordingly.")

So you never need a big-bang migration. Choose per function, based on what that function
actually does.

---

## Part 5 — Timeouts

Async makes it cheap to hold thousands of open waits. Without timeouts that's a liability: a hung
upstream accumulates paused tasks until you run out of memory.

In [ ]:
async def slow_upstream():
    await asyncio.sleep(5)
    return "finally done"

async def with_timeout():
    try:
        return await asyncio.wait_for(slow_upstream(), timeout=0.5)
    except asyncio.TimeoutError:
        return "gave up after 0.5s"

print("result:", await timed("call with a 0.5s timeout", with_timeout()))

For HTTP the same idea is a client argument:

```python
async with httpx2.AsyncClient(timeout=10.0) as client:     # seconds, every request
    r = await client.get(url)
```

`AsyncClient()` with no argument uses a 5-second default; `timeout=None` disables it, which is
almost always a mistake. Your Module 2 `fetch.py` passed `timeout=30` to `requests.get` for exactly
this reason.

> 🎯 **Every call that leaves your process gets a timeout.**

---

## Exercises

### Exercise 1 — Measure the difference yourself (⭐)

Write `fetch_city(name)` that sleeps 0.2 s and returns a dict. Fetch ten cities sequentially, then
with `gather`, and print both timings plus the speed-up factor.

In [ ]:
# Your code here


<details>
<summary>💡 Solution</summary>

```python
async def fetch_city(name):
    await asyncio.sleep(0.2)
    return {"city": name, "temp": 20.0}

names = [f"city-{i}" for i in range(10)]

async def seq():
    return [await fetch_city(n) for n in names]

async def con():
    return await asyncio.gather(*(fetch_city(n) for n in names))

with Stopwatch() as s1:
    await seq()
with Stopwatch() as s2:
    await con()

print(f"sequential {s1.elapsed:.2f}s")
print(f"gathered   {s2.elapsed:.2f}s")
print(f"speed-up   {s1.elapsed / s2.elapsed:.1f}x")
```

You should see roughly 2.0 s versus 0.2 s — about a 10x speed-up, which is not a coincidence: with
ten independent waits, the concurrent version takes as long as the *slowest single* call.
</details>

### Exercise 2 — Find the bug (⭐⭐)

Three handlers below. One will freeze the service under load. Identify it, explain precisely what
happens, and fix it **two different ways**.

```python
@app.get("/one")
async def one():
    return {"rows": db.query("SELECT 1")}          # db is psycopg2-backed, blocking

@app.get("/two")
async def two():
    async with httpx2.AsyncClient() as c:
        r = await c.get("https://example.com")
    return {"status": r.status_code}

@app.get("/three")
def three():
    return {"rows": db.query("SELECT 1")}
```

In [ ]:
# Your answer here


<details>
<summary>💡 Solution</summary>

**`/one` is the bug.** `db.query` is psycopg2-backed and therefore blocking, and it is being called
**directly inside an `async def`** with no `await`. For the whole duration of that query the event
loop is frozen: no other request is served, health checks time out, and one user's slow query becomes
everyone's outage.

`/two` is correct — an async client, properly awaited. `/three` is correct — blocking code in a
plain `def`, which FastAPI runs in a threadpool.

**Fix 1 — change one keyword** (the right answer here):

```python
@app.get("/one")
def one():                                   # was: async def
    return {"rows": db.query("SELECT 1")}
```

**Fix 2 — keep it async, push the blocking call off the loop:**

```python
from starlette.concurrency import run_in_threadpool

@app.get("/one")
async def one():
    rows = await run_in_threadpool(db.query, "SELECT 1")
    return {"rows": rows}
```

Fix 2 matters when the handler has *other* things to await as well. If it doesn't, Fix 1 is simpler
and does exactly the same thing — FastAPI's threadpool is precisely what `run_in_threadpool` uses.

(A third option is a genuinely async driver like `asyncpg`, but that's a change of database library,
not a change of handler.)
</details>

### Exercise 3 — Prove the threadpool is finite (⭐⭐⭐)

`def` handlers run in a threadpool, which is safe but **not unlimited**. Fire 50 concurrent requests
at a `def` handler that sleeps 0.5 s and measure total wall time. What does that tell you about the
pool size, and why does it matter for your database connection pool?

In [ ]:
# Your code here


<details>
<summary>💡 Solution</summary>

```python
pool_app = FastAPI()

@pool_app.get("/slow")
def slow():
    time.sleep(0.5)
    return {"ok": True}

pc = TestClient(pool_app)

with Stopwatch() as sw:
    with ThreadPoolExecutor(max_workers=50) as pool:
        list(pool.map(lambda _: pc.get("/slow"), range(50)))

print(f"50 requests x 0.5s took {sw.elapsed:.2f}s")
print(f"implies roughly {50 * 0.5 / sw.elapsed:.0f} ran concurrently")
```

If the pool were unlimited you'd see ~0.5 s. You'll see noticeably more, because the threadpool has a
finite size (AnyIO's default is 40 threads). Requests beyond that **queue**.

**Why it matters for the database:** each of those concurrent handlers checks out a connection.
Day 4 set `pool_size=5, max_overflow=5` — ten connections. Once 10 handlers are mid-query, the
eleventh waits for a connection, on top of waiting for a thread.

The full arithmetic from Day 7 and 8:

```
connections = (pool_size + max_overflow) x workers x containers
```

and that must stay under Postgres's `max_connections` (100 by default). This is why the project runs
**one worker per container** and scales by adding containers instead.
</details>

---

## ✅ Before you move on

- What does async actually eliminate — work, or waiting?
- Concurrency or parallelism: which does `asyncio` give you, and on how many threads?
- Which is dangerous: blocking code in `def`, or blocking code in `async def`? Why?
- Why does `insight-api` use `def` handlers, and is that a compromise?
- What must every outbound call have?

Next: **[Day 7 theory](../day7-docker-for-apps/13_theory_production_dockerfile.md)** — the app moves
into a food truck.